In [7]:
import csv, os, sys
import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [8]:
D = "."
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42, "font.family": "DejaVu Sans", "font.size": 8,
                     "axes.labelsize": 8, "axes.titlesize": 8.5, "xtick.labelsize": 7.5, "ytick.labelsize": 7.5,
                     "legend.fontsize": 7.5, "hatch.linewidth": 0.6})
Q = list(csv.DictReader(open(os.path.join(D, "rq2_fig3_quality.csv"))))
P = list(csv.DictReader(open(os.path.join(D, "rq2_fig4_perf.csv"))))
STY = {"Alg2": dict(label="SidePart, Alg. 2 ($k$=4)", color="#9ecae1", edgecolor="#3182bd", linewidth=0.5),
       "Alg3": dict(label="SidePart, Alg. 3 ($T$=5000)", color="#08519c", edgecolor="#08519c", linewidth=0.5),
       "Random(eps=0)": dict(label=r"SidePart, $\epsilon$=0: random placement ($k$=4)", facecolor="white",
                             edgecolor="#ff7f0e", hatch="/////", linewidth=0.5)}
# Privacy budgets on the x-axis.  None = every epsilon in the CSV.
# For linearly spaced categories set e.g. EPS_SHOW = [1, 2, 3, 4, 5, 6, 7, 8]; every value must have been
# run (it must appear in BOTH rq2_fig3_quality.csv and rq2_fig4_perf.csv), otherwise it is reported and skipped.
EPS_SHOW = None          # the linear grid 0.5, 1, 1.5, 2 is all the CSV holds
EPS_ALL = sorted({float(r["epsilon"]) for r in Q} & {float(r["epsilon"]) for r in P if r["placement"] != "Random(eps=0)"})
if EPS_SHOW is None:
    EPS = EPS_ALL
else:
    missing = [e for e in EPS_SHOW if float(e) not in EPS_ALL]
    if missing:
        print("epsilon not in the data (run these first), skipped:", missing)
    EPS = [float(e) for e in EPS_SHOW if float(e) in EPS_ALL]
step = np.diff(EPS)
print("epsilon on the x-axis:", [f"{e:g}" for e in EPS],
      "(linearly spaced)" if len(EPS) > 2 and np.allclose(step, step[0]) else "(NOT linearly spaced)")

epsilon on the x-axis: ['0.5', '1', '1.5', '2'] (linearly spaced)


In [9]:
def style(ax):
    ax.grid(True, axis="y", alpha=0.3, lw=0.5); ax.set_axisbelow(True)
    for sp in ax.spines.values(): sp.set_linewidth(0.6)
    ax.tick_params(width=0.6, length=2.5)
def save(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(D, f"{name}.{ext}"), dpi=300, bbox_inches="tight", pad_inches=0.01)
    plt.close(fig)


In [10]:
# ── Figure 3: layout quality — one function per panel, used by the combined and the separate figures ─
COMBINED, SEPARATE = False, True          # only one file per panel (set COMBINED = True for the 2x2 figures)
SEP_SIZE = (3.4, 2.4)                     # single-column size of a separate panel

def qv(alg, col):
    return [float(next(r for r in Q if r["algorithm"] == alg and float(r["epsilon"]) == e)[col]) for e in EPS]
# (a) = 1 - Rand index vs. the noiseless layout (rq2::layoutChurn): share of record PAIRS whose
#     co-location differs — not a share of rows, hence the label.
panels = [("row_assignment_error_pct_mean", "row_assignment_error_pct_sd", "Pairs co-located differently (%)",
           "Layout change vs. noiseless layout", "row_error"),
          ("partition_size_cv", None, "Partition-size CV", "Partition-size imbalance", "size_cv"),
          ("partitions_mean", None, "Partitions $K$", "Number of partitions $K$ (line = min–max)", "partitions"),
          ("dp_padding_pct", None, "DP padding / real rows (%)", "DP padding / real rows (%)", "dp_padding")]

def quality_panel(ax, col, sd, ylab, title):
    x = np.arange(len(EPS)); w = 0.38
    for i, alg in enumerate(("Alg2", "Alg3")):
        m = qv(alg, col)
        # error bar = mean +- sd, but its lower end is clipped at 0 (a negative error rate is impossible;
        # an unclipped bar would also pull the y-axis below 0)
        yerr = [[min(s, v) for v, s in zip(m, qv(alg, sd))], qv(alg, sd)] if sd else None
        ax.bar(x + (i - 0.5) * w, m, w, yerr=yerr, capsize=1, error_kw=dict(lw=0.5), **STY[alg])
    if col == "partitions_mean":
        lo, hi = qv("Alg3", "partitions_min"), qv("Alg3", "partitions_max")
        for j in range(len(EPS)):
            if hi[j] > lo[j]: ax.plot([x[j] + w / 2] * 2, [lo[j], hi[j]], color="black", lw=0.7)
    ax.set_xticks(x); ax.set_xticklabels([f"{e:g}" for e in EPS], fontsize=6.5)
    ax.set_xlabel(r"Privacy budget $\epsilon$"); ax.set_ylabel(ylab); ax.set_title(title); style(ax)
    ax.set_ylim(bottom=0)                 # every metric here is >= 0: start the y-axis at the x-axis

if COMBINED:
    fig, axs = plt.subplots(2, 2, figsize=(7.0, 4.2))
    for ax, (col, sd, ylab, title, tag), letter in zip(axs.flat, panels, "abcd"):
        quality_panel(ax, col, sd, ylab, f"({letter}) {title}")
    h, l = axs[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.03))
    fig.tight_layout(pad=0.3, rect=(0, 0, 1, 0.96)); save(fig, "rq2_fig3_quality")
if SEPARATE:
    for col, sd, ylab, title, tag in panels:
        fig, ax = plt.subplots(figsize=SEP_SIZE)
        quality_panel(ax, col, sd, ylab, title)
        ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.10), ncol=2, frameon=False,
                  borderaxespad=0.15, handlelength=1.5, fontsize=6.5, columnspacing=0.8)
        fig.tight_layout(pad=0.2); save(fig, f"rq2_fig3_{tag}")
print("figure 3 done")


figure 3 done


In [11]:
# ── Figure 4: runtime at two loads; epsilon = 0 random placement as its own group ─
def pv(load, place, e, col):
    for r in P:
        if int(r["offered_txn_s"]) == load and r["placement"] == place and float(r["epsilon"]) == e:
            return float(r[col])
    return None
PERF = [(load, metric, ylab, title, tag)
        for load in (300, 1000)
        for metric, ylab, title, tag in (
            ("throughput_txn_s_incl_repartitioning", "Committed txn/s", f"Throughput, {load} txn/s offered", "throughput"),
            ("repartitioning_pause_pct", "Repartitioning pause (%)", f"Repartitioning overhead, {load} txn/s", "repart"))]

SHOW_REPART_COUNTS = False   # True = print the number of repartitionings above each overhead bar

def perf_panel(ax, load, metric, ylab, title):
    x = np.arange(len(EPS) + 1); w = 0.38
    rv = pv(load, "Random(eps=0)", 0.0, metric)              # epsilon = 0: one bar in the first group
    ax.bar([x[0]], [rv], w, **STY["Random(eps=0)"])
    if metric.startswith("throughput"):
        t = pv(load, "Random(eps=0)", 0.0, "throughput_txn_s")
        ax.plot([x[0] - w / 2, x[0] + w / 2], [t, t], color="black", lw=0.9)
    elif SHOW_REPART_COUNTS:
        ax.text(x[0], rv + 0.4, str(int(pv(load, "Random(eps=0)", 0.0, "repartitionings"))),
                ha="center", va="bottom", fontsize=6.5)
    for i, alg in enumerate(("Alg2", "Alg3")):
        xs = x[1:] + (i - 0.5) * w
        ys = [pv(load, alg, e, metric) for e in EPS]
        ax.bar(xs, ys, w, **STY[alg])
        for xi, e, y in zip(xs, EPS, ys):
            if metric.startswith("throughput"):
                t = pv(load, alg, e, "throughput_txn_s")
                ax.plot([xi - w / 2, xi + w / 2], [t, t], color="black", lw=0.9,
                        label="excluding repartitioning pauses" if (alg == "Alg3" and e == EPS[0]) else None)
            elif SHOW_REPART_COUNTS:
                ax.text(xi, y + 0.4, str(int(pv(load, alg, e, "repartitionings"))), ha="center",
                        va="bottom", fontsize=6.5)
    ax.axvline(0.5, color="gray", lw=0.6, ls="--")
    ax.set_xticks(x); ax.set_xticklabels([r"0" + "\n(random)"] + [f"{e:g}" for e in EPS], fontsize=6.5)
    ax.set_xlabel(r"Privacy budget $\epsilon$"); ax.set_ylabel(ylab); ax.set_title(title, fontsize=8)
    if not metric.startswith("throughput"):   # room for the count labels; 1000 txn/s pauses reach ~35%
        top = max([rv] + [pv(load, a, e, metric) for a in ("Alg2", "Alg3") for e in EPS])
        ax.set_ylim(0, max(19, top * 1.15))
    style(ax)

def perf_legend(target, axes, ncol, **kw):
    h, l = [], []
    for ax in axes:
        for a, b in zip(*ax.get_legend_handles_labels()):
            if b not in l: h.append(a); l.append(b)
    for k in ("Random(eps=0)", "Alg2", "Alg3"):          # bar entries (bars carry no label)
        patch = {("facecolor" if q == "color" else q): v for q, v in STY[k].items() if q != "label"}
        h.append(plt.Rectangle((0, 0), 1, 1, **patch))
        l.append(STY[k]["label"])
    keys = [STY[k]["label"] for k in ("Random(eps=0)", "Alg2", "Alg3")] + \
           [s for s in ("excluding repartitioning pauses",) if s in l]
    target.legend([h[l.index(s)] for s in keys], keys, ncol=ncol, frameon=False, **kw)

if COMBINED:
    fig, axs = plt.subplots(2, 2, figsize=(7.0, 4.6))
    for ax, (load, metric, ylab, title, tag), letter in zip(axs.flat, PERF, "abcd"):
        perf_panel(ax, load, metric, ylab, f"({letter}) {title}")
    perf_legend(fig, axs[0], 2, loc="upper center", bbox_to_anchor=(0.5, 1.04))
    fig.tight_layout(pad=0.3, rect=(0, 0, 1, 0.95)); save(fig, "rq2_fig4_perf")
if SEPARATE:
    for load, metric, ylab, title, tag in PERF:
        fig, ax = plt.subplots(figsize=SEP_SIZE)
        perf_panel(ax, load, metric, ylab, title)
        perf_legend(ax, [ax], 2, loc="lower center", bbox_to_anchor=(0.5, 1.12), borderaxespad=0.15,
                    handlelength=1.4, fontsize=5.8, columnspacing=0.8, handletextpad=0.4)
        fig.tight_layout(pad=0.2); save(fig, f"rq2_fig4_{tag}_{load}")
print("figure 4 done")


figure 4 done
